# Offline evaluation of adaptive engines for the LEGO forklift assembly AUI

This notebook reproduces and extends the offline evaluation reported in the manuscript
*"LLMs as Calibration-Free Adaptive Engines for Industrial Training Interfaces"*.
It compares **three adaptive engines** that decide, at each assembly step, which of five
instruction formats to display initially, and it is computed entirely from the recorded
interaction logs (`all_20_experiments.csv`).

The three engines form an **information ladder**, which is the backbone of the analysis:

| Engine | Information used | Role |
|---|---|---|
| **Deterministic reference** | population statistics only | reproduces standard practice; validation anchor |
| **Within-session threshold (Det+Hist)** | population **+** the current user's own history, via a hand-designed rule | primary, information-matched baseline |
| **LLM-based** | the same information, via a natural-language prompt | the engine under study |

Comparing **reference → Det+Hist** isolates the contribution of *within-session information*;
comparing **Det+Hist → LLM** isolates the contribution of *how the adaptation logic is expressed*
(a hand-designed rule vs. a natural-language prompt). This is what lets us attribute any
improvement to the right cause.

Run the cells top to bottom. Each operation is preceded by a short explanation, and every
statistical test is accompanied by its motivation and how to read the result.

## 1. Setup and configuration

We rely only on `pandas`, `numpy`, and `scipy.stats`. The configuration block exposes the
analysis parameters as plain variables (instead of command-line arguments), so they are easy
to change and re-run:

- `WARMUP = 3` — the **exploration warm-up**: the within-session engine ignores the user's own
  history for the first three occurrences of each step type (see §3 below).
- `K_VALUES` / `K_PRIMARY` — the **pseudo-count** sweep for the shrinkage blend; `K_PRIMARY`
  is the value used in the detailed per-format / per-step-type tables.
- `THETA = 0.5` — the decision threshold shared by both threshold-based engines.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# --- configuration ---------------------------------------------------------
CSV_PATH   = "all_20_experiments.csv"   # raw interaction logs (one row per user x step)
ACC_PATH   = "accuracy.csv"             # original per-user Det + Gemini LOO accuracies
WARMUP     = 3                          # exploration window (same-type occurrences)
K_VALUES   = [1, 2, 4, 8]               # pseudo-count sweep for Det+Hist
K_PRIMARY  = 4                          # k featured in detailed tables
THETA      = 0.5                        # decision threshold
pd.set_option("display.width", 200)

## 2. Task definition (manuscript Table 1)

The assembly procedure has **16 steps** of three types — **Picking** (7), **Assembly** (8),
and **Quality Control** (1) — and five instruction formats. Not every format is available at
every step type; structurally-absent formats are excluded from prediction *and* from the
accuracy denominator, so the metric is never inflated by "free" correct zeros.

We also encode two task constraints used during inference:

- **Format availability** by step type (the set $\mathcal{F}_s$).
- **Short/Long-text mutual exclusivity**: the two text formats cannot be shown together; when
  both clear the threshold, only the higher-scoring one is kept, and on an exact tie **short
  text is preferred** (the same rule given to the LLM in its prompt, which keeps the engine
  fully deterministic).

In [2]:
# CSV column -> manuscript format name
FORMAT_COLS = {
    "short_text_viewed":    "Short text",
    "long_text_viewed":     "Long text",
    "single_pieces_viewed": "Component image",
    "assembly_viewed":      "Assembly image",
    "video_viewed":         "Video",
}
FORMATS = list(FORMAT_COLS.keys())
SHORT, LONG = "short_text_viewed", "long_text_viewed"

# Format availability by step type (the set F_s)
AVAILABILITY = {
    "Picking":  {"short_text_viewed", "single_pieces_viewed"},
    "Assembly": set(FORMATS),
    "QC":       {"short_text_viewed", "long_text_viewed",
                 "single_pieces_viewed", "video_viewed"},
}

# Published reference values (deterministic engine + volatility), for validation
PAPER_DET = {
    "overall_mean": 74.9, "overall_sd": 11.5, "min": 57.5, "max": 96.3,
    "per_format": {
        "Short text": (69.7, 17.5, 12.8), "Long text": (82.2, 0.0, 17.8),
        "Component image": (75.0, 18.1, 6.9), "Assembly image": (93.1, 6.9, 0.0),
        "Video": (77.2, 7.2, 15.6)},
    "per_step_type": {"Picking": 68.2, "Assembly": 82.2, "QC": 62.5},
}
PAPER_VOLATILITY = {"mean": 16.2, "sd": 7.4, "min": 9, "max": 32}

## 3. Load the interaction logs and check integrity

The log is semicolon-delimited with one row per (participant, step). Before trusting any
downstream number we verify the data is well-formed: binary format columns, exactly 16 steps
per participant, and — crucially — that **structurally-absent formats are never accessed**
(e.g. no video on a picking step). If any of these fail, the analysis would be meaningless,
so we assert them explicitly.

In [3]:
def step_type(name):
    if name.startswith("Withdraw"): return "Picking"
    if name.startswith("Assembly"): return "Assembly"
    if name.startswith("Final"):    return "QC"
    raise ValueError(f"Unrecognised step_name: {name!r}")

def load(path):
    df = pd.read_csv(path, sep=";")
    df["stype"] = df["step_name"].map(step_type)
    # integrity checks
    for f in FORMATS:
        assert set(df[f].unique()) <= {0, 1}, f"{f} is not binary"
    assert (df.groupby("experiment_id").size() == 16).all(), "not 16 steps per participant"
    for st, sub in df.groupby("stype"):
        for f in FORMATS:
            if f not in AVAILABILITY[st]:
                assert not (sub[f] == 1).any(), f"absent format {f} accessed on {st}"
    return df

df = load(CSV_PATH)
print(f"Loaded {len(df)} rows / {df['experiment_id'].nunique()} participants")
print("Steps per type:", df.groupby('stype')['step_id'].nunique().to_dict())
df.head()

Loaded 320 rows / 20 participants
Steps per type: {'Assembly': 8, 'Picking': 7, 'QC': 1}


,experiment_id,action,step_id,step_name,short_text_viewed,long_text_viewed,single_pieces_viewed,assembly_viewed,video_viewed,stype
0,1,navigate_next,1,Withdraw Components for PIECE 1,1,0,1,0,0,Picking
1,1,navigate_next,2,Assembly PIECE 1,1,0,0,0,1,Assembly
2,1,navigate_next,3,Withdraw Components for PIECE 2,0,0,1,0,0,Picking
3,1,navigate_next,4,Assembly PIECE 2,0,0,0,0,1,Assembly
4,1,navigate_next,5,Withdraw Components for PIECE 3,0,0,1,0,0,Picking


## 4. Behavioural volatility index

**Why we compute it.** Per-user accuracy varies a lot across participants. We want a single,
interpretable descriptor of *how erratic* each participant's behaviour is, so that later
(§9) we can test whether an engine's accuracy depends on it. This is the quantity that makes
"behavioural heterogeneity" concrete and measurable.

**Definition (graded).** For each participant, the index is the **mean normalised Hamming
distance** between the format-access vectors of consecutive steps *of the same type*. For each
consecutive same-type pair, the distance is the number of formats whose access status changed,
divided by the number of formats available at that step type; the index averages this over all
such pairs. It is *graded*: flipping one format counts less than changing the whole accessed
set. QC has a single step, so it contributes no transitions.

**How to read it.** Higher = more erratic behaviour between repetitions of a step type; lower =
a stable, repeated pattern.

In [4]:
def volatility_index(df):
    out = {}
    for u, sub in df.groupby("experiment_id"):
        num = den = 0.0
        for st, ss in sub.groupby("stype"):
            ss = ss.sort_values("step_id")
            cols = sorted(AVAILABILITY[st]); k = len(cols)
            vecs = ss[cols].values
            for i in range(1, len(vecs)):
                num += (vecs[i] != vecs[i - 1]).sum() / k
                den += 1
        out[u] = num / den if den else np.nan
    return pd.Series(out, name="volatility")

vol = volatility_index(df)
print(f"Volatility: mean {100*vol.mean():.1f}%  SD {100*vol.std(ddof=1):.1f}%  "
      f"range {100*vol.min():.1f}-{100*vol.max():.1f}%")
print(f"(paper: mean {PAPER_VOLATILITY['mean']}%, SD {PAPER_VOLATILITY['sd']}%, "
      f"range {PAPER_VOLATILITY['min']}-{PAPER_VOLATILITY['max']}%)")

Volatility: mean 15.0%  SD 8.5%  range 3.1-32.3%
(paper: mean 16.2%, SD 7.4%, range 9-32%)


## 5. Prediction primitives

Two building blocks shared by all engines:

- `population_proportions(train)` — for each step, the fraction of the *calibration* users who
  accessed each format. This is the only signal the deterministic reference uses, and the prior
  that the within-session engine shrinks towards.
- `predict_step(scores, available)` — turns per-format real-valued scores into a binary
  visibility decision: show a format iff its score $\ge \theta$, then enforce short/long
  mutual exclusivity with the deterministic **prefer-short** tie-break.

In [37]:
def population_proportions(train):
    return {sid: {f: sub[f].mean() for f in FORMATS}
            for sid, sub in train.groupby("step_id")}

def predict_step(scores, available, diag):
    pred = {f: int(scores[f] >= THETA) for f in available}
    if SHORT in available and LONG in available and pred[SHORT] == 1 and pred[LONG] == 1:
        diag["mutex_conflicts"] += 1
        s_sc, l_sc = scores[SHORT], scores[LONG]
        if l_sc > s_sc:
            pred[SHORT] = 0
        else:                                  # s_sc > l_sc, or exact tie -> prefer short
            if abs(s_sc - l_sc) < 1e-12:
                diag["mutex_exact_ties"] += 1
            pred[LONG] = 0
    return pred

## 6. The three engines

**Deterministic reference.** Shows format $f$ at step $s$ iff the population access proportion
$\hat p(s,f) \ge \theta$. Population-only; cannot react to the individual within a session.

**Within-session threshold (Det+Hist).** Blends the population proportion with the held-out
user's *own* access rate over prior same-type steps, by evidence-weighted (Dirichlet-multinomial)
shrinkage:

$$\hat p_{\text{blend}}(s,f)=\frac{n\,\hat p_{\text{self}}(s,f)+k\,\hat p(s,f)}{n+k},\qquad
  \hat y(s,f)=\mathbf 1[\hat p_{\text{blend}}(s,f)\ge\theta].$$

Here $n$ is the number of prior same-type steps and $k$ the pseudo-count (prior strength).
Two design choices make it a *fair* foil for the LLM and keep it interpretable:

1. **Exploration warm-up:** while $n < $ `WARMUP` (the first three same-type occurrences), the
   engine uses the population proportion only — it assumes the operator is still exploring. The
   same assumption is expressed to the LLM in its prompt.
2. **Nesting:** when $n < $ `WARMUP` (and always on the single QC step), the blend reduces
   *exactly* to the deterministic reference, so Det+Hist generalises the reference rather than
   replacing it.

Note Det+Hist uses only the user's *access* history (not prediction-outcome feedback), so if
anything it is given slightly *less* than the LLM — a conservative, LLM-favouring choice.

In [6]:
def run_deterministic(test_user_df, p_pop, diag):
    preds = {}
    for _, row in test_user_df.sort_values("step_id").iterrows():
        sid, st = row["step_id"], row["stype"]
        avail = AVAILABILITY[st]
        preds[sid] = predict_step({f: p_pop[sid][f] for f in avail}, avail, diag)
    return preds

def run_det_hist(test_user_df, p_pop, k, warmup, diag):
    preds = {}
    hist = {st: [] for st in AVAILABILITY}          # accumulating same-type access vectors
    for _, row in test_user_df.sort_values("step_id").iterrows():
        sid, st = row["step_id"], row["stype"]
        avail = AVAILABILITY[st]
        n = len(hist[st])
        scores = {}
        for f in avail:
            p_pop_f = p_pop[sid][f]
            if n < warmup:
                blended = p_pop_f                    # exploration: population only (nests Det)
            else:
                p_self_f = np.mean([h[f] for h in hist[st]])
                blended = (n * p_self_f + k * p_pop_f) / (n + k)
                if abs(blended - THETA) < 1e-12:
                    diag["score_exact_half"] += 1
            scores[f] = blended
        preds[sid] = predict_step(scores, avail, diag)
        hist[st].append({f: int(row[f]) for f in FORMATS})   # update AFTER predicting
    return preds

## 7. Leave-one-out evaluation and the accuracy metric

**Why leave-one-out (LOO).** With only $N=20$ participants, LOO maximises the calibration data
at each fold (19 users) while keeping the test participant entirely unseen, preventing leakage.
It is deterministic for both threshold engines (no random splits) and yields a *per-participant*
accuracy, which is exactly what the dispersion and correlation analyses need.

**Accuracy metric.** For each (step, user), accuracy is the fraction of correct binary
predictions over the *available* formats only. Each format decision has roughly symmetric error
costs (showing an unneeded format clutters the screen; hiding a needed one costs an extra tap),
so a symmetric metric that also rewards correctly hiding a format is appropriate. Per-user
accuracy is the mean over the 16 steps; overall accuracy is the mean over all (user, step) cells.

In [7]:
def step_accuracy(pred_step, truth_row, available):
    return np.mean([int(pred_step[f] == int(truth_row[f])) for f in available])

def evaluate(df, engine_fn, **kwargs):
    diag = {"mutex_conflicts": 0, "mutex_exact_ties": 0, "score_exact_half": 0}
    per_user, cells = [], []
    for u in sorted(df["experiment_id"].unique()):
        train = df[df["experiment_id"] != u]
        test  = df[df["experiment_id"] == u]
        p_pop = population_proportions(train)
        preds = engine_fn(test, p_pop, diag=diag, **kwargs)
        accs = []
        for _, row in test.iterrows():
            sid, st = row["step_id"], row["stype"]; avail = AVAILABILITY[st]
            accs.append(step_accuracy(preds[sid], row, avail))
            for f in avail:
                yh, y = preds[sid][f], int(row[f])
                cells.append({"experiment_id": u, "step_id": sid, "stype": st, "format": f,
                              "correct": int(yh == y), "fp": int(yh == 1 and y == 0),
                              "fn": int(yh == 0 and y == 1)})
        per_user.append({"experiment_id": u, "accuracy": np.mean(accs)})
    return pd.DataFrame(per_user), pd.DataFrame(cells), diag

# small reporting helpers
def overall_stats(per_user):
    a = per_user["accuracy"].values * 100
    return dict(mean=a.mean(), sd=a.std(ddof=1), min=a.min(), max=a.max(), vals=a)

def per_format_table(cells):
    rows = []
    for col, name in FORMAT_COLS.items():
        sub = cells[cells["format"] == col]
        rows.append({"Format": name, "n": len(sub), "Acc": 100*sub["correct"].mean(),
                     "FP": 100*sub["fp"].mean(), "FN": 100*sub["fn"].mean()})
    return pd.DataFrame(rows).round(1)

def per_step_type_table(cells):
    rows = []
    for st in ["Picking", "Assembly", "QC"]:
        sub = cells[cells["stype"] == st]
        g = sub.groupby(["experiment_id", "step_id"])["correct"].mean()
        rows.append({"Step type": st, "n_cells": len(sub), "Acc": 100*g.mean()})
    return pd.DataFrame(rows).round(1)

## 8. Deterministic reference — the validation anchor

Before extending anything, we confirm the pipeline is correct by reproducing the published
deterministic-engine numbers. If our overall accuracy, per-format error rates, and per-step-type
accuracies match the manuscript **to the decimal**, the LOO loop, the accuracy metric, and the
constraint handling are all verified — and any Det+Hist number from the *same* pipeline inherits
that trust. Watch for the bracketed `[paper]` values to coincide with ours.

In [8]:
det_user, det_cells, det_diag = evaluate(df, run_deterministic)
det = overall_stats(det_user)
print(f"Deterministic: overall {det['mean']:.1f}%  SD {det['sd']:.1f}%  "
      f"min {det['min']:.1f}%  max {det['max']:.1f}%")
print(f"paper        : overall {PAPER_DET['overall_mean']}%  SD {PAPER_DET['overall_sd']}%  "
      f"min {PAPER_DET['min']}%  max {PAPER_DET['max']}%")
print("\nPer-format (ours):"); display(per_format_table(det_cells))
print("Per-step-type (ours):"); display(per_step_type_table(det_cells))
print("tie diagnostics:", det_diag)

Deterministic: overall 74.9%  SD 11.5%  min 57.5%  max 96.2%
paper        : overall 74.9%  SD 11.5%  min 57.5%  max 96.3%

Per-format (ours):


,Format,n,Acc,FP,FN
0,Short text,320,69.7,17.5,12.8
1,Long text,180,82.2,0.0,17.8
2,Component image,320,75.0,18.1,6.9
3,Assembly image,160,93.1,6.9,0.0
4,Video,180,77.2,7.2,15.6


Per-step-type (ours):


,Step type,n_cells,Acc
0,Picking,280,68.2
1,Assembly,800,82.2
2,QC,80,62.5


tie diagnostics: {'mutex_conflicts': 0, 'mutex_exact_ties': 0, 'score_exact_half': 0}


## 9. Within-session threshold (Det+Hist) — the $k$ sweep

We now run the primary baseline across the pseudo-count sweep `K_VALUES`. Reporting a sweep
(rather than a single tuned $k$) is deliberate: it shows the conclusions do **not** depend on a
fortunate parameter choice, which is important because $k$ is the one free parameter of this
engine. For each $k$ we record the mean/SD and, looking ahead to §10, the variance ratio against
the reference and the paired test against it.

We expect the mean to sit around **81%** across $k\in\{1,2,4,8\}$ and the qualitative picture to
be stable. `exact-ties = 0` throughout confirms the prefer-short tie-break essentially never
fires, so the engine is effectively deterministic regardless.

In [9]:
def f_test_var(a, b):
    va, vb = np.var(a, ddof=1), np.var(b, ddof=1)
    if va >= vb: F, dfn, dfd = va/vb, len(a)-1, len(b)-1
    else:        F, dfn, dfd = vb/va, len(b)-1, len(a)-1
    p = 2*min(stats.f.cdf(F, dfn, dfd), 1-stats.f.cdf(F, dfn, dfd))
    return F, p

dethist_by_k = {}
rows = []
for k in K_VALUES:
    dh_user, dh_cells, dh_diag = evaluate(df, run_det_hist, k=k, warmup=WARMUP)
    dethist_by_k[k] = (dh_user, dh_cells, dh_diag)
    dh = overall_stats(dh_user)
    F, Fp = f_test_var(det["vals"], dh["vals"])
    w, wp = stats.wilcoxon(det_user["accuracy"], dh_user["accuracy"])
    rows.append({"k": k, "mean": round(dh["mean"],1), "SD": round(dh["sd"],1),
                 "min": round(dh["min"],1), "max": round(dh["max"],1),
                 "F_vs_det": round(F,2), "F_p": round(Fp,3),
                 "Wilcoxon_p_vs_det": round(wp,3),
                 "exact_ties": dh_diag["mutex_exact_ties"]})
pd.DataFrame(rows)

/home/vincenzocutrona/.local/lib/python3.10/site-packages/scipy/stats/_morestats.py:3414: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "


,k,mean,SD,min,max,F_vs_det,F_p,Wilcoxon_p_vs_det,exact_ties
0,1,81.4,7.5,68.4,97.5,2.35,0.070,0.003,0
1,2,81.4,7.5,66.6,97.5,2.33,0.072,0.001,0
2,4,81.3,7.6,65.9,97.5,2.28,0.080,0.001,0
3,8,79.2,8.6,62.2,97.5,1.80,0.211,0.006,0


## 10. The key statistical comparisons (A, A′, B)

We now load the original per-user accuracies for the deterministic engine and the **LLM**
(Gemini, mean of 3 simulation runs) from `accuracy.csv`, and run the three comparisons that
carry the paper's argument. We use the Det+Hist results at `K_PRIMARY`.

For every comparison we report four complementary quantities. Their **motivation** and
**interpretation**:

- **Paired Wilcoxon signed-rank test** *(primary).* The two engines are measured on the *same*
  20 participants, so the values are paired. With $N=20$ and no guarantee that the per-user
  differences are normally distributed, a non-parametric paired test is the safe primary choice:
  it ranks the absolute differences and asks whether their signs are balanced around zero (null:
  no systematic difference). **Read it as:** $p<0.05$ ⇒ the engines differ systematically; a
  large $p$ ⇒ we *cannot detect* a difference (with this small sample this is **not** proof of
  equivalence).
- **Paired $t$-test** *(cross-check).* Parametric companion; if it agrees with Wilcoxon the
  conclusion is robust to the distributional assumption.
- **95% confidence interval of the mean difference** *(effect size).* A $p$-value alone hides
  magnitude and precision. **Read it as:** the plausible range of the true mean gap; if it
  **includes 0** the difference is not significant at $\alpha=0.05$, and its width shows how
  (under-)powered the comparison is.
- **$F$-test for equality of variances** *(uniformity).* The paper's claim is partly about
  *serving a heterogeneous population evenly*, i.e. the **spread** of per-user accuracy, not its
  mean. The $F$-test compares the two variances (we report $F=\text{larger}/\text{smaller}$).
  **Read it as:** significant ⇒ one engine's per-user spread is genuinely larger; the
  smaller-variance engine is more *uniform*. (The $F$-test is sensitive to non-normality, so we
  treat it as indicative.)

The three comparisons map onto the information ladder:

- **A — reference vs Det+Hist:** does *within-session information* help? (expect: yes, significant)
- **A′ — reference vs LLM:** the original paper's headline comparison.
- **B — Det+Hist vs LLM:** does the *LLM's natural-language reasoning* add anything beyond a
  hand-designed rule with the same information? (expect: no detectable difference)

In [10]:
acc = pd.read_csv(ACC_PATH); acc.columns = ["experiment_id", "det_paper", "llm"]
dh_user, dh_cells, dh_diag = dethist_by_k[K_PRIMARY]
m = (det_user.rename(columns={"accuracy": "det"})
     .merge(dh_user.rename(columns={"accuracy": "dethist"}), on="experiment_id")
     .merge(acc[["experiment_id", "llm"]], on="experiment_id"))

def compare(a, b, label):
    diff = (a - b) * 100
    w, wp = stats.wilcoxon(a, b)
    t, tp = stats.ttest_rel(a, b)
    se = diff.std(ddof=1) / np.sqrt(len(diff))
    ci = stats.t.ppf(0.975, len(diff)-1) * se
    F, Fp = f_test_var(a.values, b.values)
    return {"comparison": label, "mean_diff_pp": round(diff.mean(),2),
            "Wilcoxon_p": round(wp,3), "t_p": round(tp,3),
            "CI95_low": round(diff.mean()-ci,1), "CI95_high": round(diff.mean()+ci,1),
            "F_var": round(F,2), "F_p": round(Fp,3)}

results = pd.DataFrame([
    compare(m["dethist"], m["det"], "A  reference -> Det+Hist"),
    compare(m["llm"],     m["det"], "A' reference -> LLM"),
    compare(m["dethist"], m["llm"], "B  Det+Hist vs LLM"),
])
print("Means:  reference %.1f%%   Det+Hist %.1f%%   LLM %.1f%%" %
      (100*m['det'].mean(), 100*m['dethist'].mean(), 100*m['llm'].mean()))
print("Wins:  Det+Hist>ref %d/20   Det+Hist>LLM %d/20   LLM>ref %d/20" %
      ((m['dethist']>m['det']).sum(), (m['dethist']>m['llm']).sum(), (m['llm']>m['det']).sum()))
results

Means:  reference 74.9%   Det+Hist 81.3%   LLM 79.7%
Wins:  Det+Hist>ref 15/20   Det+Hist>LLM 13/20   LLM>ref 12/20


,comparison,mean_diff_pp,Wilcoxon_p,t_p,CI95_low,CI95_high,F_var,F_p
0,A reference -> Det+Hist,6.44,0.001,0.001,3.2,9.7,2.28,0.080
1,A' reference -> LLM,4.82,0.090,0.061,-0.2,9.9,4.27,0.003
2,B Det+Hist vs LLM,1.62,0.189,0.226,-1.1,4.3,1.87,0.182


**What to conclude from §10.** Comparison **A** is significant (within-session information
genuinely improves accuracy over standard practice). Comparison **B** is *not* significant on
either the mean or the variance: the LLM and the hand-designed within-session rule are
statistically indistinguishable. Together this localises the gain to *within-session
conditioning*, not to the LLM's reasoning — so the LLM's value is that it delivers that
conditioning **calibration-free** (no hand-designed rule, no $k$ to tune), not that it predicts
better.

> **A note on "significant vs. not significant".** A′ (reference→LLM) reaches significance on
> variance while A (reference→Det+Hist) does not, yet B (Det+Hist vs LLM) shows the two are not
> significantly different from each other. The difference between "significant" and
> "non-significant" is not itself a significant difference; with $N=20$ we simply cannot resolve
> whether the LLM is *more uniform* than the hand-designed rule. This is stated as a limitation
> rather than a claim.

## 11. Does volatility explain per-user accuracy? (Pearson correlation)

**Why.** §4 gave us a per-user volatility score; here we test whether it *predicts* how well
each engine serves a participant. The informative pattern is the **contrast across engines**.

**Test.** Pearson's $r$ between volatility and per-user accuracy, with significance from
$t=r\sqrt{(n-2)/(1-r^2)}$ on $n-2$ degrees of freedom.

**How to read it.** A strong **negative** $r$ means erratic users are predicted worse. We expect
this for the two within-session engines (they condition on a noisy individual history) and a
**near-zero** $r$ for the deterministic reference (it ignores the individual entirely). That
asymmetry is the point: it shows volatility-sensitivity is a property of *within-session
conditioning in general*, not something specific to the LLM.

In [11]:
def corr(y_col):
    r, p = stats.pearsonr(mv["volatility"], mv[y_col])
    t = r * np.sqrt((len(mv)-2)/(1-r**2))
    return {"engine": y_col, "pearson_r": round(r,2), "t(18)": round(t,2), "p": round(p,4)}

mv = m.merge(vol.reset_index().rename(columns={"index": "experiment_id"}), on="experiment_id")
pd.DataFrame([corr("det"), corr("dethist"), corr("llm")])

,engine,pearson_r,t(18),p
0,det,-0.08,-0.32,0.7506
1,dethist,-0.56,-2.88,0.0099
2,llm,-0.80,-5.57,0.0000


## 12. Per-format and per-step-type detail (Det+Hist at `K_PRIMARY`)

Finally, the disaggregated view used in the paper's Tables 3–4.

**What to look for.**
- *Component image* is where the LLM is weakest (high false-positive rate); the Det+Hist rule,
  given the same information, does **not** share this over-prediction — evidence the failure is
  specific to the LLM's interpretation, not to within-session conditioning.
- *Picking* steps (only two formats) are where within-session conditioning helps most; *Assembly*
  steps (five formats) are where the hand-designed rule does best and the LLM loses ground.

In [12]:
dh_user, dh_cells, dh_diag = dethist_by_k[K_PRIMARY]
print(f"Det+Hist per-format (k={K_PRIMARY}):"); display(per_format_table(dh_cells))
print(f"Det+Hist per-step-type (k={K_PRIMARY}):"); display(per_step_type_table(dh_cells))

Det+Hist per-format (k=4):


,Format,n,Acc,FP,FN
0,Short text,320,79.7,11.2,9.1
1,Long text,180,85.6,0.0,14.4
2,Component image,320,79.4,14.1,6.6
3,Assembly image,160,93.1,6.9,0.0
4,Video,180,73.9,5.0,21.1


Det+Hist per-step-type (k=4):


,Step type,n_cells,Acc
0,Picking,280,81.8
1,Assembly,800,83.2
2,QC,80,62.5


## 13. Summary

Reading the ladder bottom-up:

1. **Within-session information matters** — Det+Hist significantly beats the population-only
   reference (comparison A).
2. **The LLM's reasoning adds nothing detectable on top** — Det+Hist and the LLM are
   statistically indistinguishable on accuracy *and* variance (comparison B), even though the LLM
   was given slightly more information (prediction-outcome feedback).
3. **Therefore the LLM's contribution is at design time** — it reaches the performance of an
   expertly hand-tuned within-session rule **without** anyone designing or calibrating that rule
   (no blending formula, no warm-up length, no $k$). That calibration-free specification, plus
   the within-session co-adaptive loop both engines support, is the human–AI symbiosis the paper
   argues for.

All numbers for the deterministic and LLM engines reproduce the original study; the Det+Hist
numbers come from this notebook and should be cross-checked against an independent implementation
before submission.